In [13]:
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Load your trained FinBERT aspect model
model_path = "./aspect_extractor_model_finbert"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

# Load your inference pipeline
aspect_pipeline = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Load your input JSON file (containing sentences)
with open("sentences_only.json", "r", encoding="utf-8") as f:
    data = json.load(f)

results = []

for entry in data:
    sentence = entry["content"]
    preds = aspect_pipeline(sentence)

    # Filter only LABEL_1 (aspects)
    aspects = [
        {"word": p["word"], "entity_group": p["entity_group"], "score": float(p["score"])}
        for p in preds
        if p["entity_group"] == "LABEL_1"
    ]

    # Create a simpler output structure
    results.append({
        "sentence": sentence,
        "predicted_aspects": aspects
    })

# Save cleaned results to a new JSON file
with open("filtered_aspects_output.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("✅ Filtered aspects saved to filtered_aspects_output.json")


✅ Filtered aspects saved to filtered_aspects_output.json


In [10]:
import numpy as np

def convert_numpy_types(obj):
    """Recursively convert numpy types to Python types."""
    if isinstance(obj, dict):
        return {k: convert_numpy_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(v) for v in obj]
    elif isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    else:
        return obj

# Convert all NumPy values before saving
output_data_cleaned = convert_numpy_types(output_data)

with open("aspect_predictions_output.json", "w", encoding="utf-8") as f:
    json.dump(output_data_cleaned, f, indent=4, ensure_ascii=False)

print("✅ Cleaned and saved as aspect_predictions_output.json")

✅ Cleaned and saved as aspect_predictions_output.json
